# Predicting CANDOR survey responses based on growth rates $\alpha$

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from scipy.stats import chisquare, skewtest, t

# stats in R
import rpy2.robjects as ro
from rpy2.robjects import Formula
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri, default_converter, conversion

import warnings; warnings.filterwarnings('ignore')

In [2]:
vars = [
    ### Affective motive
    'best_affect', 'conv_leader', 'conversationalist', 'how_enjoyable', 'i_like_you', 'i_felt_close_to_my_partner', 'i_think_your_status', 'i_would_like_to_become_friends', 'overall_affect',
    'responsive_1','responsive_2','responsive_3',
    'you_are_good_listener', 'you_are_kind', 'you_are_quickwitted','you_are_warm','you_like_me',
    #test affectives!
    'you_are_intelligent', 'you_are_friendly', 'in_common',

    ### Cognitive motives
    'anticipated_each_other_sr6', 'developed_joint_perspective_sr2', 'thoughts_became_more_alike_sr5',
    #test cognitives!
    'shared_reality', 'our_thoughts_synced_up_sr1', 'saw_world_in_same_way_sr8',
]

Data paths used in the project

In [3]:
subcorpora_col = 'corpus'

SLOPES_PATH = '../../EcoPsych/english/4-POWER-LAW-SCALING/reports/all.csv'

candor_meta_data_path = '../../updated-data/all-candor-meta_data.csv'

TABLES_FIGS_PATH = 'tables_and_figures/'
if not os.path.exists(TABLES_FIGS_PATH):
    os.mkdir(TABLES_FIGS_PATH)

PARAMETER_ESTIMATES_PATH = 'params/'
if not os.path.exists(PARAMETER_ESTIMATES_PATH):
    os.mkdir(PARAMETER_ESTIMATES_PATH)

if not os.path.exists(os.path.join(TABLES_FIGS_PATH, 'individual_var_stats')):
    os.mkdir(os.path.join(TABLES_FIGS_PATH, 'individual_var_stats'))

## Importing data

In [4]:
results = pd.read_csv(SLOPES_PATH)
results.head()

,corpus,cid,speaker,self,a,b,k
0,CABNC,CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUG,False,3.156620e-02,-0.516245,118
1,CABNC,CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUG,True,2.093744e+00,-0.681666,10
2,CABNC,CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUN,False,1.560238e-04,-0.506909,61
3,CABNC,CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-KBWPSUN,True,7.756877e-08,4.656958,3
4,CABNC,CABNC-KBWRE01F-KBW,CABNC-CABNC-KBWRE01F-KBW-PS087,False,2.607473e-04,-0.100337,194


##### Some preliminary statistics

In [ ]:
print((results['b'].abs() == np.inf).sum(), (results['b'].abs() == np.inf).mean())
# results.isna().sum()

### Removing null-test results/bad values

In [5]:
sel = results['b'].isna() | (results['b'].abs() == np.inf)
sel.sum(), sel.mean(), (~sel).sum()

(np.int64(0), np.float64(0.0), np.int64(23876))

In [6]:
# is_null = np.array(['null-' in x for x in tqdm(results['x_turn_id'])])
# is_null = results['null']
# is_null.sum()

In [7]:
results = results.loc[
    (~sel)
    # & (~is_null)
    & results[subcorpora_col].isin(['CANDOR'])
]
results.shape

(6612, 7)

In [8]:
results['conversation_id'] = ['-'.join(it.split('-')[2:]) for it in tqdm(results['cid'])]

  0%|          | 0/6612 [00:00<?, ?it/s]

In [9]:
results.head()

,corpus,cid,speaker,self,a,b,k,conversation_id
32,CANDOR,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,5ee9201596dd421333942d06,False,0.000042,-0.473086,10440,5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec
33,CANDOR,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,5ee9201596dd421333942d06,True,0.000079,-0.593119,10296,5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec
34,CANDOR,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,5f91a4aeb9f8bc031e8c27c9,False,0.000048,-0.474464,10296,5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec
35,CANDOR,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,5f91a4aeb9f8bc031e8c27c9,True,0.000041,-0.439823,10296,5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec
36,CANDOR,CANDOR-9-97087c78-d2a1-4959-b72a-80f858a0afac,5d1554e28ecfae0017a026eb,False,0.000445,-0.571810,12720,97087c78-d2a1-4959-b72a-80f858a0afac


## Merging with CANDOR metadata

In [10]:
dfm = pd.read_csv(candor_meta_data_path)[['user_id', 'convo_id']+vars]
print(dfm.shape)


(3312, 28)


rescaling data to be in same scale.

In [11]:
for col in tqdm(list(dfm)[2:]):
    dfm[col] = (dfm[col] / dfm[col].max() ) * 10

  0%|          | 0/26 [00:00<?, ?it/s]

In [12]:
dfm.head()

,user_id,convo_id,best_affect,conv_leader,conversationalist,how_enjoyable,i_like_you,i_felt_close_to_my_partner,i_think_your_status,i_would_like_to_become_friends,...,you_like_me,you_are_intelligent,you_are_friendly,in_common,anticipated_each_other_sr6,developed_joint_perspective_sr2,thoughts_became_more_alike_sr5,shared_reality,our_thoughts_synced_up_sr1,saw_world_in_same_way_sr8
0,5efbe4f0c9f37c1a3b70afaf,56f21a75-4e86-4dee-9a15-a4f68e86e1a3,7.777778,6.666667,7.6,7.777778,8.571429,NaN,8.0,NaN,...,8.571429,8.888889,10.000000,7.777778,7.142857,10.000000,8.571429,7.678571,7.142857,5.714286
1,5f26378609aca228dcfd19fa,56f21a75-4e86-4dee-9a15-a4f68e86e1a3,7.777778,7.777778,7.3,7.777778,7.142857,NaN,6.0,NaN,...,8.571429,6.666667,10.000000,8.888889,8.571429,7.142857,5.714286,8.392857,8.571429,10.000000
2,5d8e361e6e920e0014b459d9,94a1abaa-d89e-4d0d-82ad-1c8de8155a2f,7.777778,3.333333,3.3,5.555556,5.714286,NaN,4.0,NaN,...,5.714286,5.555556,2.222222,1.111111,1.428571,1.428571,2.857143,3.392857,1.428571,2.857143
3,5f4d435c030fef9d0136b496,94a1abaa-d89e-4d0d-82ad-1c8de8155a2f,5.555556,8.888889,6.5,4.444444,7.142857,NaN,6.0,NaN,...,7.142857,10.000000,10.000000,3.333333,1.428571,2.857143,1.428571,2.857143,2.857143,1.428571
4,5ca6bbf13b5fcf00100996e9,4e3233b7-5d89-47e9-a795-25452f99a1c9,10.000000,4.444444,9.6,10.000000,10.000000,8.571429,6.0,8.571429,...,10.000000,8.888889,10.000000,6.666667,7.142857,4.285714,7.142857,6.428571,5.714286,5.714286


#### Testing for normalcy and uniformity

In [ ]:
normalcy_and_other_stats = []

In [ ]:
for var in vars:
    sel = dfm[var].isna()
    maxima = dfm[var].loc[~sel].max()
    minima = dfm[var].loc[~sel].min()
    medi = dfm[var].loc[~sel].median()
    mu = dfm[var].loc[~sel].mean()
    mode = dfm[var].loc[~sel].mode()
    skew_stat, p_skew = skewtest(dfm[var].loc[~sel].values)

    dv = dfm[var].loc[~sel].value_counts()
    chi_stat, p_chi = chisquare(f_obs=dv.values, f_exp=np.array([1/len(dv)]*len(dv))*dv.sum())

    normalcy_and_other_stats += [
        {
            'variable': var,
            'range': '[{}, {}]'.format(minima, maxima),
            'median': medi.item(),
            'mean': mu.item(),
            'mode': mode.values,
            'skew_stat': skew_stat, 'skew_p': p_skew,
            'chi_stat': chi_stat, 'chi_p': p_chi
        }
    ]

normalcy_and_other_stats = pd.DataFrame(normalcy_and_other_stats)

In [ ]:
normalcy_and_other_stats.head(n=40)

In [ ]:
normalcy_and_other_stats.to_csv('candor-normalcy-stats.csv', index=False, encoding='utf-8')

#### Merging docs

In [13]:
results = pd.merge(
    left=results, left_on=['speaker', 'conversation_id'],
    right=dfm, right_on=['user_id', 'convo_id'],
    # left=results, left_on=['speaker'],
    # right=dfm, right_on=['user_id'],
    how='left'
)

In [14]:
results = results.drop(columns=['user_id', 'convo_id'])
# results = results.drop(columns=['user_id'])
# results.columns
results.isna().sum()

corpus                                0
cid                                   0
speaker                               0
self                                  0
a                                     0
b                                     0
k                                     0
conversation_id                       0
best_affect                         110
conv_leader                         110
conversationalist                   110
how_enjoyable                       110
i_like_you                          110
i_felt_close_to_my_partner         4972
i_think_your_status                 112
i_would_like_to_become_friends     4972
overall_affect                      110
responsive_1                        112
responsive_2                        112
responsive_3                        112
you_are_good_listener               110
you_are_kind                        110
you_are_quickwitted                 110
you_are_warm                        110
you_like_me                         110


## Predicting individual survey variables from decay rates $\alpha$

### Converting categorical cols to numbered indexes

In [15]:
## convert speaker to integers
col = 'speaker'
vs = results[col].unique()
# sid = {sp:i+1 for i,sp in enumerate(np.random.choice(vs,size=(len(vs),),replace=False))}
sid = {sp:i+1 for i,sp in enumerate(vs)}
results[col+'_'] = [sid[sp] for sp in tqdm(results[col])]

  0%|          | 0/6612 [00:00<?, ?it/s]

In [16]:
## convert turn_ids to integers
col = 'conversation_id'
vs = results[col].astype(str).unique()
# sid = {sp:i+1 for i,sp in enumerate(np.random.choice(vs,size=(len(vs),),replace=False))}
sid = {sp:i+1 for i,sp in enumerate(vs)}
results[col+'_'] = [sid[sp] for sp in tqdm(results[col].astype(str))]

  0%|          | 0/6612 [00:00<?, ?it/s]

### Running JAGS model in R

In [17]:
results['other'] = (~results['self']).astype(int)
results['self'] = results['self'].astype(int)
print(results[['self', 'other']].head(n=2))

   self  other
0     0      1
1     1      0


In [18]:
# model_ = str(model_script)

for var in tqdm(vars):
    sel_2 = ~results[var].isna()

    with localconverter(default_converter + pandas2ri.converter):
        df_ = conversion.py2rpy(results[['b', 'conversation_id_', 'speaker_', 'self', 'other', var]].loc[sel_2])

    ro.globalenv['df'] = df_

    ro.r("""
    library(R2jags)
    # print(colnames(df))
    survey_var <- df${}
    slope <- df$b
    b0 <- mean(slope)
    b0range <- abs(b0) *.25
    speaker <- df$speaker_
    conversation <- df$conversation_id_
    is_self <- df$self
    is_other <- df$other
    KSPEAKER <- max(speaker)
    CCONVERSATION <- max(conversation)
    # print(KSPEAKER)
    # print(CCONVERSATION)
    ROWS <- length(slope)

    sb0 <- mean(slope[is_self==1])
    ob0 <- mean(slope[is_other==1])
    sv <- var(slope[is_self==1])
    ov <- var(slope[is_other==1])
    # print(sb0)
    # print(ob0)

    data <- list(
        # 'b0',
        # 'b0range',
        'sv', 'ov',
        'survey_var', 'is_self', 'is_other', 'slope', 'speaker',
        # 'conversation',
        'sb0', 'ob0',
        # 'CCONVERSATION',
        'KSPEAKER',
        'ROWS')

    myinits <- list(
        list(
            # sigma=c(rep(.01, KSPEAKER)),
            gamma=.5,
            eta=.5,
            # omega=1.01,
            # b0=0.1,
            # self_b0=0.0,
            # other_b0=0.0
            # self_b0=-0.001,
            # other_b0=-0.001,
            beta=0.99,
            phi=0.1
        )
    )

    parameters <- c(
        'beta',
        # 'gamma',
        # 'eta',
        # 'omega',
        'phi',
        'self_b0',
        'other_b0'
    )

    samples<- jags(data, inits=myinits, parameters, model.file='model.txt', n.chains=1, n.iter=5000, n.burnin=100, DIC=T, progress.bar="text")

    beta <- samples$BUGSoutput$sims.list$beta
    # sigma <- samples$BUGSoutput$sims.list$sigma
    # b0 <- samples$BUGSoutput$sims.list$b0
    self_b0 <- samples$BUGSoutput$sims.list$self_b0
    other_b0 <- samples$BUGSoutput$sims.list$other_b0
    phi <- samples$BUGSoutput$sims.list$phi
    # omega <- samples$BUGSoutput$sims.list$omega
    save(
        beta,
        # b0,
        # slope,
        # is_self,
        # is_other,
        self_b0,
        other_b0,
        phi,
        # omega,
        file='{}/{}.RData'
    )
    """.format(var, PARAMETER_ESTIMATES_PATH, var))

  0%|          | 0/26 [00:00<?, ?it/s]

R callback write-console: Loading required package: rjags
  
R callback write-console: Loading required package: coda
  
R callback write-console: Linked to JAGS 4.3.2
  
R callback write-console: Loaded modules: basemod,bugs
  
R callback write-console: 
Attaching package: ‘R2jags’

  
R callback write-console: The following object is masked from ‘package:coda’:

    traceplot

  
R callback write-console: module glm loaded
  


Compiling model graph
   Resolving undeclared variables
   Allocating nodes
Graph information:
   Observed stochastic nodes: 6502
   Unobserved stochastic nodes: 2916
   Total graph size: 49795

Initializing model

  |**************************************************| 100%
  |**************************************************| 100%
Compiling model graph
   Resolving undeclared variables
   Allocating nodes
Graph information:
   Observed stochastic nodes: 6502
   Unobserved stochastic nodes: 2916
   Total graph size: 49795

Initializing model

  |**************************************************| 100%
  |**************************************************| 100%
Compiling model graph
   Resolving undeclared variables
   Allocating nodes
Graph information:
   Observed stochastic nodes: 6502
   Unobserved stochastic nodes: 2916
   Total graph size: 50151

Initializing model

  |**************************************************| 100%
  |**************************************************| 